# Семинар 3. Языковая модель и генерация

На лекциях мы разобрали, как устроено языковое моделирование, как модель оценивает вероятность текста и как из распределения вероятностей получить конкретную последовательность токенов. Сегодня соберем все эти идеи вместе и посмотрим, что происходит, когда они встречаются с реальными данными.

## Что мы сделаем

1. **Разберем языковую модель.** Вспомним, как модель оценивает вероятность последовательности и как работает chain rule.
2. **Построим n-граммную модель.** Обучим биграммную модель, столкнемся с разреженностью и попробуем исправить ее с помощью сглаживания.
3. **Поиграем с генерацией.** Сравним greedy и beam search и увидим, почему выбор наиболее вероятных токенов может приводить к повторам и зацикливанию.
4. **Покрутим стратегии декодирования.** Исследуем влияние температуры, top-k и top-p на случайность и разнообразие генерации.
5. **Оценим качество.** Посчитаем перплексию для моделей разного порядка и обсудим, почему более вероятный текст не обязательно оказывается более хорошим.
6. **(Опционально) Сравним с GPT.** Применим те же стратегии декодирования к предобученной нейросетевой языковой модели и сравним результаты.

## Как устроено занятие

Мы будем последовательно проходить по ячейкам, запускать эксперименты и обсуждать результаты. Мы почти не будем заниматься обучением нейросетей - это тема ДЗ. Вместо этого возьмем маленькую и прозрачную n-граммную модель, в которой можно увидеть и проверить каждую вероятность. На ней разберем все основные механизмы генерации из лекции 2: от greedy и beam search до температуры, top-k и top-p.


## Блок 0. Подготовка

Начнем с постановки из лекции 1:

> Языковая модель задает **распределение вероятностей над последовательностями токенов**. Она отвечает на вопрос: насколько вероятно такое продолжение с учетом уже известного контекста?

Но сама модель не генерирует текст. Она только сообщает вероятности. А вот какой токен выбрать из этого распределения - уже задача стратегии декодирования.

Сегодня специально разделим эти две части: сначала научимся получать и понимать вероятности, затем будем экспериментировать с разными способами выбора токенов.



In [ ]:
import numpy as np
import re
import random
from collections import Counter, defaultdict
import matplotlib.pyplot as plt

np.random.seed(42)

# служебные токены начала и конца предложения
BOS, EOS = "<s>", "</s>"
print("Окружение готово. numpy:", np.__version__)

### Данные

Для семинара возьмем нарочно узкий жанр - кулинарные инструкции. Он хорошо подходит для экспериментов с простой языковой моделью:

- жанр достаточно **шаблонный**, поэтому n-граммная модель сможет выучить характерные последовательности;
- словарь относительно небольшой - вероятности можно **посмотреть и понять**, а не потеряться среди тысяч слов;
- ошибки генерации легко заметить глазами: фразы вроде "нарежьте духовку соломкой" сразу показывают, что статистически правдоподобная последовательность не обязательно имеет смысл.

Чтобы ноутбук был полностью самодостаточным, корпус соберем из шаблонов фраз.



In [ ]:
def build_corpus(seed=0):
    """Комбинаторно собираем корпус кулинарных инструкций."""
    verbs_prep = ["нарежьте", "измельчите", "натрите", "порубите", "очистите"]
    verbs_heat = ["обжарьте", "потушите", "проварите", "запеките", "припустите"]
    ingredients = ["лук", "морковь", "картофель", "чеснок", "перец", "кабачок", "грибы", "помидоры", "мясо", "курицу", "рыбу", "капусту"]
    manner = ["мелкими кубиками", "тонкими ломтиками", "крупными кусками", "соломкой", "кольцами"]
    places = ["на сковороде", "в кастрюле", "на противне", "в казане", "в глубокой миске"]
    times = ["пять минут", "десять минут", "пятнадцать минут", "полчаса", "до готовности", "до золотистой корочки", "до мягкости"]
    spices = ["соль", "перец", "паприку", "тимьян", "куркуму", "базилик"]
    liquids = ["воду", "бульон", "сливки", "молоко", "вино", "сметану"]

    sents = []
    for v in verbs_prep:
        for ing in ingredients:
            for m in manner:
                sents.append(f"{v} {ing} {m}")
    for v in verbs_heat:
        for ing in ingredients:
            for pl in places:
                for t in times:
                    sents.append(f"{v} {ing} {pl} {t}")
    for sp in spices:
        sents.append(f"добавьте {sp} по вкусу")
        sents.append(f"приправьте блюдо {sp}")
    for lq in liquids:
        for t in times:
            sents.append(f"влейте {lq} и тушите {t}")
    # немного "сюжетных" фраз, чтобы текст был живее
    extra = [
        "разогрейте духовку до двухсот градусов",
        "доведите до кипения и убавьте огонь",
        "накройте крышкой и оставьте настояться",
        "перемешайте все ингредиенты до однородности",
        "подавайте блюдо горячим с зеленью",
        "остудите и украсьте перед подачей"
    ]
    sents += extra * 20

    # Частотные конструкции для эксперимента с температурой.
    # Некоторые продолжения будут встречаться чаще других.
    common = (
        ["обжарьте лук на сковороде"] * 8
        + ["обжарьте чеснок на сковороде"] * 4
        + ["обжарьте морковь на сковороде"] * 2
        + ["обжарьте грибы на сковороде"] * 1

        + ["добавьте соль по вкусу"] * 8
        + ["добавьте перец по вкусу"] * 3
        + ["добавьте паприку по вкусу"] * 1

        + ["влейте воду и тушите десять минут"] * 6
        + ["влейте бульон и тушите десять минут"] * 3
        + ["влейте сливки и тушите десять минут"] * 1
    )
    sents += common

    random.seed(seed)
    random.shuffle(sents)
    return sents

raw_sentences = build_corpus()
print("Всего предложений в корпусе:", len(raw_sentences))
print("\nПримеры:")
for s in raw_sentences[:6]:
    print(s)

In [ ]:
def tokenize(line):
    """Нижний регистр + только кириллица (пунктуацию выкидываем)."""
    return re.findall(r"[а-яё]+", line.lower())

sentences = [tokenize(s) for s in raw_sentences]

all_tokens = [w for s in sentences for w in s]
vocab_counter = Counter(all_tokens)

print("Всего токенов:", len(all_tokens))
print("Уникальных токенов (|V|):", len(vocab_counter))
print("Топ-10 самых частых:")
for w, c in vocab_counter.most_common(10):
    print(f"{w} - {c}")

## Блок 1. Языковая модель как распределение: вероятность фразы

Начнем с самого простого варианта - униграммной модели, которая полностью игнорирует контекст. Для каждого слова оцениваем вероятность по его частоте в корпусе:

$$P(w) = \frac{N(w)}{\sum_u N(u)}$$

Здесь:
- $w$ - конкретное слово, вероятность которого мы хотим оценить;
- $N(w)$ - сколько раз это слово встретилось в корпусе;
- $\sum_u N(u)$ - общее количество слов в корпусе, то есть сумма частот всех слов.

Затем попробуем с помощью этих вероятностей оценить, насколько типичной для нашего корпуса выглядит целая фраза.




In [ ]:
unigram_prob = {w: c / len(all_tokens) for w, c in vocab_counter.items()}

# отсортируем и покажем несколько
for w in ["лук", "обжарьте", "духовку", "минут"]:
    print(f"P({w}) = {unigram_prob.get(w, 0):.4f}")

Теперь перейдем от отдельных слов к целой фразе. Самое простое предположение - считать слова независимыми и перемножать их вероятности:

$$
P(w_1,\dots,w_n) = \prod_{i=1}^{n} P(w_i)
$$

Но здесь возникает практическая проблема: вероятности отдельных слов меньше 1, поэтому при умножении даже нескольких десятков значений получается очень маленькое число, с которым неудобно работать.

Поэтому вместо самой вероятности обычно используют логарифм вероятности. Логарифм превращает произведение в сумму:

$$
\log P(w_1,\dots,w_n)
= \log \prod_{i=1}^{n} P(w_i)
= \sum_{i=1}^{n} \log P(w_i)
$$

То есть вместо того, чтобы перемножать множество маленьких чисел, мы складываем их логарифмы. Это удобнее и с точки зрения вычислений, и для дальнейшего сравнения фраз: чем выше log-probability, тем более вероятной считает фразу модель.

> Важно: это не меняет саму модель и не меняет порядок фраз по вероятности. Мы просто используем более удобную численную форму представления той же вероятности.



In [ ]:
def unigram_logprob(text, prob, unk=1e-6):
    """Лог-вероятность фразы в предположении независимости слов (униграмма)."""
    tokens = tokenize(text) if isinstance(text, str) else text
    lp = 0.0
    for w in tokens:
        lp += np.log(prob.get(w, unk)) # unk — заглушка для незнакомых слов
    return lp

good = "обжарьте лук на сковороде"
weird = "обжарьте пять минут духовку" # слова корпусные, но порядок бредовый

print(f"log P({good}) = {unigram_logprob(good, unigram_prob):.2f}")
print(f"log P({weird}) = {unigram_logprob(weird, unigram_prob):.2f}")

**Что важно заметить.**

Обе фразы состоят из слов, которые встречались в корпусе, поэтому униграммная модель оценивает их примерно одинаково. Но одна фраза может быть осмысленной, а другая - совершенно бессмысленной.

Причина в том, что униграмма не учитывает контекст и порядок слов. Для нее текст - практически просто набор отдельных слов. Чтобы учитывать, какое слово обычно следует за предыдущими, нужно перейти к условным вероятностям и цепному правилу:

$$
P(y_1,\dots,y_n) = \prod_{t=1}^{n} P(y_t \mid y_{<t})
$$

Здесь $y_{<t}$ - все слова, которые стоят перед $y_t$. То есть вероятность каждого следующего слова зависит от уже известного контекста.

**Вопрос на подумать**

Цепное правило точное, это не приближение. Почему же тогда нельзя применить его "в лоб" и оценивать каждую $P(y_t \mid y_{<t})$ прямым подсчетом по всему предыдущему контексту?

**Возможный ответ**

Проблема в разреженности. Полная последовательность предыдущих слов почти всегда уникальна: такой контекст в обучающем корпусе, скорее всего, вообще не встретится. Тогда нужную условную вероятность невозможно надежно оценить прямым подсчетом.

Например, если мы хотим оценить $P(\text{соль} \mid \text{добавьте в миску и аккуратно перемешайте})$, нужно знать, сколько раз именно такой полный контекст встречался в корпусе. Для реального текста почти наверняка получим нулевой или слишком маленький счетчик.

Поэтому приходится упрощать контекст: вместо всех предыдущих слов учитывать только несколько последних. Это и есть марковское допущение, на котором строятся n-граммные модели.



## Блок 2. n-граммная модель: учитываем контекст и ловим разреженность

Теперь добавим контекст. В лекции мы обсуждали марковское допущение: вместо всей истории будем учитывать только несколько последних слов:

$$
P(y_t \mid y_1,\dots,y_{t-1})
\approx
P(y_t \mid y_{t-n+1},\dots,y_{t-1})
$$

То есть для n-граммной модели важно не все предложение целиком, а только последние $n-1$ токенов.

Вероятность следующего токена оцениваем по его частоте после такого контекста:

$$
P(y_t \mid \text{context})
\frac{N(\text{context},,y_t)}
{N(\text{context})}
$$

Здесь $N(\text{context}, y_t)$ - сколько раз в корпусе встречалась такая последовательность контекста и следующего слова, а $N(\text{context})$ - сколько раз вообще встречался этот контекст.

Например, для биграммной модели контекст состоит из одного слова:

$$
P(\text{соль}\mid\text{добавьте})
\frac{N(\text{добавьте, соль})}
{N(\text{добавьте})}
$$

Так модель уже учитывает порядок слов: вероятность "соль после добавьте" может быть высокой, даже если вероятность слова "соль" сама по себе невелика.

Напишем один класс `NGramLM`, который будет работать с любым $n$: униграмма при `n=1`, биграмма при `n=2`, триграмма при `n=3` и так далее.

Это будет наш "черный ящик" на весь семинар: дальше мы будем менять способы генерации и оценки, не меняя саму языковую модель.



In [ ]:
class NGramLM:
    """Простая n-граммная языковая модель на счетчиках частот."""
    def __init__(self, sentences, n=2):
        self.n = n
        self.context_counts = Counter() # N(context)
        self.ngram_counts = Counter() # N(context, w)
        vocab = set()
        for s in sentences:
            padded = [BOS] * (n - 1) + s + [EOS]
            vocab.update(padded)
            for i in range(len(padded) - n + 1):
                context = tuple(padded[i : i + n - 1])
                word = padded[i + n - 1]
                self.context_counts[context] += 1
                self.ngram_counts[(context, word)] += 1
        self.vocab = sorted(vocab)

    def _ctx(self, history):
        """Обрезаем историю до последних n-1 токенов (марковское допущение)."""
        return tuple(history[-(self.n - 1):]) if self.n > 1 else tuple()

    def prob(self, word, history):
        """P(word | history) без сглаживания."""
        ctx = self._ctx(history)
        if self.context_counts[ctx] == 0:
            return 0.0
        return self.ngram_counts[(ctx, word)] / self.context_counts[ctx]

    def next_word_dist(self, history):
        """Всё распределение P(· | history) как dict {слово: вероятность}."""
        ctx = self._ctx(history)
        total = self.context_counts[ctx]
        if total == 0:
            return {}
        return {w: c / total
                for (c_ctx, w), c in self.ngram_counts.items() if c_ctx == ctx}

bigram = NGramLM(sentences, n=2)
print("Биграммная модель обучена. |V| =", len(bigram.vocab))

Посмотрим, какие слова модель считает наиболее вероятными в начале предложения. Это распределение

$$
P(y_1 \mid \text{начало})
$$

то есть вероятность каждого возможного первого слова с учетом специального токена начала `<s>`.

Именно из этого распределения модель будет выбирать первое слово при генерации. Дальше, после каждого выбранного слова, контекст изменится и модель построит новое распределение вероятностей для следующего токена. Так шаг за шагом и будет формироваться весь текст.



In [ ]:
dist = bigram.next_word_dist([BOS])
top = sorted(dist.items(), key=lambda x: -x[1])[:8]

print(f"Сумма вероятностей по всему словарю: {sum(dist.values()):.4f} (должна быть 1.0)")
print("Чем чаще всего начинается инструкция:")
for w, p in top:
    print(f"{w} {p:.3f}")

Теперь вернемся к нашей паре фраз из блока 1, но оценим их с помощью **биграммной модели**. В отличие от униграммы, она учитывает предыдущее слово и поэтому может различать не только состав фразы, но и порядок слов.

Посмотрим, станет ли осмысленная фраза более вероятной, чем бессмысленная, и насколько сильно контекст меняет оценку модели.


In [ ]:
def ngram_logprob(model, text, unk_logprob=np.log(1e-8)):
    """Лог-вероятность фразы по n-граммной модели (без сглаживания)."""
    tokens = tokenize(text) if isinstance(text, str) else text
    padded = [BOS] * (model.n - 1) + tokens + [EOS]
    lp = 0.0
    for i in range(model.n - 1, len(padded)):
        p = model.prob(padded[i], padded[:i])
        lp += np.log(p) if p > 0 else unk_logprob # нулевую вероятность заменяем заглушкой
    return lp

good = "обжарьте лук на сковороде"
weird = "обжарьте пять минут духовку"

print("Униграмма (мешок слов):")
print(f"{good} -> {unigram_logprob(good, unigram_prob):7.2f}")
print(f"{weird} -> {unigram_logprob(weird, unigram_prob):7.2f}")
print("\nБиграмма (учитывает предыдущее слово):")
print(f"{good} -> {ngram_logprob(bigram, good):7.2f}")
print(f"{weird} -> {ngram_logprob(bigram, weird):7.2f}")

**Вопрос**

У униграммы обе фразы получались примерно одинаково вероятными. Что изменилось у биграммы и почему? И обратите внимание на численную заглушку для нулевой вероятности - где именно она сработала?

**Возможный ответ**

Биграмма резко "наказала" бессмысленную фразу: в ней встретились последовательности вроде `(минут, духовку)`, которых в корпусе не было. Для такой биграммы модель получает вероятность `0`, а `log(0)` вычислить нельзя.

Поэтому в коде вместо нуля мы подставляем маленькое число `1e-8`. В логарифмической шкале это дает большой отрицательный штраф:

$$
\log(1e-8) \approx -18.4
$$

Осмысленная фраза состоит из биграмм, которые встречались в корпусе, поэтому такого штрафа не получает.

Это и есть главное преимущество контекста: модель начинает учитывать не только отдельные слова, но и то, какие слова обычно стоят рядом.

Но здесь появляется новая проблема. Мы фактически сказали: "если биграммы не было, дадим ей очень маленькую вероятность". Это пока лишь численный костыль. В лекции 1 упоминалось более содержательное решение - сглаживание.



### 2.1 Разреженность

В лекции 1 упоминалось о важной проблеме n-граммных моделей: **достаточно одной невстречавшейся n-граммы, чтобы вероятность всей последовательности стала равна нулю**.

Сейчас проверим это напрямую - уберем нашу численную заглушку и посмотрим, что произойдет с вероятностью фразы, содержащей хотя бы одну неизвестную биграмму.


In [ ]:
def raw_seq_prob(model, text):
    """Настоящая (несглаженная) вероятность фразы. Вернет 0.0, если встретится хотя бы одна неизвестная n-грамма."""
    tokens = tokenize(text) if isinstance(text, str) else text
    padded = [BOS] * (model.n - 1) + tokens + [EOS]
    p = 1.0
    for i in range(model.n - 1, len(padded)):
        p *= model.prob(padded[i], padded[:i])
    return p

for phrase in ["обжарьте лук на сковороде",
               "обжарьте лук соломкой", # каждая биграмма по отдельности известна
               "обжарьте духовку тонким слоем"]: # грамматично, но абсурд
    print(f"P = {raw_seq_prob(bigram, phrase):.3e}   {phrase}")

Третья фраза синтаксически корректна, но получает нулевую вероятность - модель считает ее невозможной.

Это не ошибка реализации, а фундаментальное свойство оценки вероятностей по частотам: **если хотя бы одна n-грамма имеет вероятность 0, вероятность всей последовательности тоже становится 0**.

Решение - **сглаживание**. Его идея проста: немного уменьшить вероятность часто встречающихся событий и перераспределить эту массу в пользу редких и даже невстречавшихся событий, чтобы нулевых вероятностей больше не возникало.



### 2.2 Сглаживание Лапласа (add-α)

Простейший способ избавиться от нулевых вероятностей - **сглаживание add-α**, или сглаживание Лапласа. Идея проста: добавляем небольшую величину $\alpha$ к счетчику **каждого возможного слова**, даже если оно ни разу не встречалось после данного контекста.

$$
P(w \mid \text{ctx}) = \frac{N(\text{ctx}, w) + \alpha}{N(\text{ctx}) + \alpha \cdot |V|}
$$

Здесь:
- $N(\text{ctx}, w)$ - сколько раз слово $w$ встречалось после контекста `ctx`;
- $N(\text{ctx})$ - сколько раз вообще встречался этот контекст;
- $|V|$ - размер словаря;
- $\alpha$ - величина сглаживания. Чем она больше, тем больше вероятностной массы мы отдаем редким и невстречавшимся словам.

Например, если некоторой биграммы в корпусе не было, при обычной оценке ее вероятность равна 0. После сглаживания она становится:

$$
P(w \mid \text{ctx}) = \frac{\alpha}{N(\text{ctx}) + \alpha |V|} > 0
$$

Теперь **ни одна вероятность не равна нулю**, а значит, одна невстречавшаяся n-грамма больше не обнуляет вероятность всей последовательности.

Реализуем сглаживание как наследника `NGramLM` - сама модель и подсчет частот останутся прежними, изменится только **формула оценки вероятности**.



In [ ]:
class LaplaceLM(NGramLM):
    """n-граммная модель со сглаживанием add-alpha."""
    def __init__(self, sentences, n=2, alpha=1.0):
        super().__init__(sentences, n=n)
        self.alpha = alpha
        self.V = len(self.vocab)

    def prob(self, word, history):
        ctx = self._ctx(history)
        return (self.ngram_counts[(ctx, word)] + self.alpha) / (self.context_counts[ctx] + self.alpha * self.V)

bigram_smooth = LaplaceLM(sentences, n=2, alpha=1.0)

print("Та же абсурдная фраза, но теперь со сглаживанием:")
print(f"без сглаживания: P = {raw_seq_prob(bigram, 'обжарьте духовку тонким слоем'):.3e}")
print(f"со сглаживанием: P = {raw_seq_prob(bigram_smooth, 'обжарьте духовку тонким слоем'):.3e}")

print("\nОсмысленная фраза для сравнения:")
print(f"со сглаживанием: P = {raw_seq_prob(bigram_smooth, 'обжарьте лук на сковороде'):.3e}")

После сглаживания абсурдная фраза больше не получает нулевую вероятность - модель теперь допускает даже невстречавшиеся биграммы. Но это не означает, что она считает их правдоподобными: абсурдная фраза все равно оказывается на несколько порядков менее вероятной, чем осмысленная.

Это именно то, чего мы хотим от сглаживания: невстречавшееся событие не обнуляет всю последовательность, но известные и правдоподобные последовательности по-прежнему получают значительно более высокую вероятность.


**Вопрос**

Как изменится решение при изменении условий? Что произойдет, если взять очень большое значение $\alpha$, например `1000`? А если постепенно приближать $\alpha$ к нулю? Сначала предположите результат, затем проверьте его экспериментально.

**Возможный ход рассуждения**

- При **$\alpha \to 0$** сглаживание практически исчезает, и модель возвращается к обычной оценке по частотам. Невстречавшиеся n-граммы снова получают вероятность, близкую к нулю.
- При **большом $\alpha$** влияние исходных счетчиков становится все слабее. Вероятности начинают выравниваться и приближаться к равномерному распределению по словарю.
- Поэтому слишком маленькое $\alpha$ почти не решает проблему разреженности, а слишком большое стирает информацию о реальных частотах.

То есть $\alpha$ задает компромисс: нужно дать некоторую вероятность невстречавшимся событиям, но не настолько большую, чтобы потерять структуру корпуса.


In [ ]:
test_phrase = "обжарьте лук на сковороде"
print(f"Вероятность фразы '{test_phrase}' при разных α:\n")
for a in [0.001, 0.1, 1.0, 100.0, 1000.0]:
    m = LaplaceLM(sentences, n=2, alpha=a)
    print(f"α = {a} P = {raw_seq_prob(m, test_phrase):.3e}")

Чем больше $\alpha$, тем сильнее сглаживание: мы все больше перераспределяем вероятность от часто встречающихся событий к редким и невстречавшимся. В пределе распределение становится все ближе к **равномерному**, поэтому информация о реальных частотах корпуса постепенно теряет значение.

При маленьком $\alpha$ модель почти возвращается к обычной оценке по частотам - она сильнее доверяет данным, но сохраняет проблему очень маленьких вероятностей для редких n-грамм. При большом $\alpha$ нулей уже нет, но чрезмерное сглаживание начинает **стирать различия между вероятными и маловероятными продолжениями**.

Поэтому $\alpha$ - это компромисс между **"доверять наблюдаемым частотам" и "не обнулять невстречавшиеся события"**. Его разумно подбирать экспериментально, например, по качеству на отложенной выборке.


## Блок 3. Генерация

До сих пор мы только оценивали готовые фразы. Теперь попробуем с помощью той же модели сгенерировать новый текст.

Процедура одинакова для любой авторегрессионной языковой модели:

1. по текущему контексту получаем распределение $P(y_t \mid y_{<t})$;
2. стратегия декодирования выбирает из него один токен;
3. добавляем выбранный токен в контекст;
4. повторяем, пока модель не сгенерирует `</s>` или не будет достигнут максимальный размер текста.

Важно: сама языковая модель на этом этапе не меняется. Мы будем менять только способ выбора токена на шаге 2. Для всех экспериментов используем одну и ту же `bigram_smooth`.

Начнем с самой простой стратегии - **greedy decoding**. На каждом шаге она выбирает токен с максимальной вероятностью.

Никакой случайности здесь нет: при одном и том же контексте всегда выбирается один и тот же токен. Поэтому `greedy` дает **детерминированную генерацию**.



In [ ]:
def generate(model, strategy="greedy", max_len=20, start=None, T=1.0, top_k=None, top_p=None, seed=None):

    if seed is not None:
        np.random.seed(seed)

    history = [BOS] * (model.n - 1)
    if start:
        history += tokenize(start)
        output = tokenize(start)[:]
    else:
        output = []

    for _ in range(max_len):
        dist = model.next_word_dist(history)
        if not dist:
            dist = {w: model.prob(w, history) for w in model.vocab} # сглаженная модель

        words = np.array(list(dist.keys()))
        probs = np.array([dist[w] for w in words], dtype=float)
        probs = probs / probs.sum()

        if strategy == "greedy":
            chosen = words[int(np.argmax(probs))]
        else: # sampling
            probs = _apply_temperature(probs, T)
            words, probs = _truncate(words, probs, top_k=top_k, top_p=top_p)
            chosen = np.random.choice(words, p=probs)

        if chosen == EOS:
            break
        output.append(chosen)
        history.append(chosen)

    return " ".join(output)


def _apply_temperature(probs, T):
    """Масштабируем распределение температурой."""
    if T == 1.0:
        return probs
    logits = np.log(np.clip(probs, 1e-12, None)) / T
    logits -= logits.max()
    e = np.exp(logits)
    return e / e.sum()


def _truncate(words, probs, top_k=None, top_p=None):
    """Обрезаем хвост распределения: top-k и/или top-p, потом перенормируем."""
    order = np.argsort(-probs)
    words, probs = words[order], probs[order]
    if top_k is not None:
        words, probs = words[:top_k], probs[:top_k]
    if top_p is not None:
        cum = np.cumsum(probs)
        cut = int(np.searchsorted(cum, top_p)) + 1
        words, probs = words[:cut], probs[:cut]
    probs = probs / probs.sum()
    return words, probs

In [ ]:
# greedy детерминирован
print("greedy, 3 запуска подряд:")
for _ in range(3):
    print(generate(bigram_smooth, strategy="greedy"))

### 3.1 Text degeneration

В лекции 2 упоминалось о **degeneration** - ситуации, когда текст формально остается допустимым, но постепенно становится бедным, однообразным и начинает повторяться.

Особенно легко это увидеть при `greedy`: модель каждый раз выбирает локально самый вероятный токен. Если после какого-то контекста одно продолжение становится особенно выгодным, модель снова и снова выбирает его, даже если в результате получается бессмысленный повтор.

Спровоцируем такой сценарий специально: добавим в корпус фразу с **самоподхватывающейся конструкцией**, где выбранное слово создает контекст, который снова делает то же слово наиболее вероятным.

Посмотрим, сможет ли `greedy` выбраться из такого цикла - или будет продолжать его до достижения ограничения по длине.



In [ ]:
loopy_corpus = sentences + [tokenize("помешивайте и помешивайте пока не загустеет")] * 60
loopy_model = LaplaceLM(loopy_corpus, n=2, alpha=0.01)

print("greedy, старт со слова 'помешивайте':\n")
print(generate(loopy_model, strategy="greedy", start="помешивайте", max_len=25))

Цикл `помешивайте и помешивайте и ...` и есть **degeneration**. Механика ровно такая, как в лекции: после `помешивайте` самым вероятным токеном становится `и`, а после `и` - снова `помешивайте`. Поскольку `greedy` всегда выбирает максимум, он не может самостоятельно выйти из этой петли.


## Блок 4. Температура

Первый способ уйти от жадного выбора - **сэмплировать** токен из распределения вероятностей, а не всегда брать `argmax`. Но если просто выбирать токены из полного распределения, модель может иногда вытягивать редкие слова из длинного хвоста и получать бессвязный текст.

**Температура** позволяет управлять тем, насколько распределение сосредоточено вокруг наиболее вероятных токенов:

$$
P_T(w) = \frac{\exp(z_w / T)}{\sum_u \exp(z_u / T)}
$$

Где $z_w$ - исходный логит токена, а $T$ - температура.

* **$T < 1$** - распределение становится "острее": вероятные токены получают еще больший вес, поведение приближается к `greedy`;
* **$T = 1$** - распределение не меняется;
* **$T > 1$** - распределение становится более равномерным: вероятность менее популярных токенов увеличивается, поэтому генерация становится разнообразнее, но возрастает риск странных продолжений.

Сначала посмотрим глазами, как температура меняет одно и то же распределение. Так будет проще понять, что именно она делает, прежде чем использовать ее для генерации текста.



In [ ]:
# берем реальное распределение из модели: что идет после "обжарьте"
base_dist = bigram_smooth.next_word_dist(["обжарьте"])
words = list(base_dist.keys())
base = np.array([base_dist[w] for w in words])
order = np.argsort(-base)[:10] # топ-10 для наглядности
words = [words[i] for i in order]
base = base[order]; base /= base.sum()

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, T in zip(axes, [0.3, 1.0, 2.5]):
    p = _apply_temperature(base, T)
    ax.bar(range(len(words)), p, color="#4C72B0")
    ax.set_title(f"T = {T}")
    ax.set_xticks(range(len(words)))
    ax.set_xticklabels(words, rotation=60, ha="right", fontsize=8)
axes[0].set_ylabel("вероятность")
fig.suptitle('Одно и то же распределение P(· | "обжарьте") при разной температуре', y=1.02)
plt.tight_layout()
plt.show()

При низкой температуре ($T=0.3$) распределение становится более "острым": почти вся вероятностная масса сосредотачивается на одном-двух наиболее вероятных токенах, и поведение приближается к `greedy`. При высокой температуре ($T=2.5$) различия между токенами сглаживаются: менее вероятные варианты получают больше массы, а распределение становится ближе к равномерному.

**Вопрос на подумать**

Значит ли это, что высокая температура всегда делает текст лучше? Что мы выигрываем в разнообразии и чем за это платим?



Теперь посмотрим, как изменение распределения проявляется уже не на графике, а в самой генерации.

In [ ]:
print("Как температура влияет на текст (модель одна и та же!):\n")
for T in [0.2, 0.7, 1.0, 2.0]:
    print(f"T = {T}")
    for i in range(2): # 2 запуска, чтобы увидеть разброс
        txt = generate(bigram_smooth, strategy="sample", T=T, seed=100 + i)
        print(txt)

**Вопрос**

Посмотрите на несколько значений $T$ и сравните результаты:

- при **$T=0.2$** - насколько отличаются два запуска? Почему случайность почти не проявляется?
- при **$T=2.0$** - генерация стала разнообразнее. Но стала ли она **качественнее**?
- какую температуру вы бы выбрали для **автоподсказок в поисковой строке**, а какую - для **генератора идей названий блюд**? Объясните свой выбор.

**Ориентир для обсуждения**

При низкой $T$ два запуска почти совпадают: распределение становится очень "острым", поэтому сэмплирование почти всегда выбирает один из самых вероятных токенов. Поведение приближается к `greedy`, но сохраняется риск однообразных повторов.

При высокой $T$ вариантов становится больше, но вместе с полезным разнообразием растет вероятность странных и маловероятных продолжений. **Разнообразнее не значит лучше.**

Поэтому для автоподсказок логично выбрать **низкую $T$** - здесь важны предсказуемость и надежность. Для генератора идей названий можно взять **$T$ около 1 или выше** - здесь разнообразие ценнее, а отдельные неудачные варианты допустимы.

Универсально лучшего значения $T$ нет: выбор зависит от того, **какой компромисс между предсказуемостью, качеством и разнообразием нужен конкретной задаче**.


Давайте проверим гипотезу из блока 3.1. Поможет ли сэмплирование разорвать тот самый цикл `помешивайте и помешивайте`?

В `greedy` мы всегда выбирали самый вероятный токен, поэтому модель застревала в одной и той же последовательности. Теперь выбор будет случайным: более вероятные токены по-прежнему выбираются чаще, но менее вероятные тоже получают шанс.

Посмотрим, достаточно ли этой небольшой смены стратегии, чтобы модель смогла выйти из цикла.



In [ ]:
print("Зацикленная модель, но теперь СЭМПЛИРОВАНИЕ вместо greedy:\n")
for i in range(3):
    print(generate(loopy_model, strategy="sample", T=1.0, start="помешивайте", max_len=20, seed=i))

Цикл разорван - и модель при этом **не изменилась**. Изменился только способ выбора следующего токена: сэмплирование иногда выбирает не самый вероятный вариант, благодаря чему генерация может выйти из петли.

Это ровно то, о чем говорилось в лекции: **повторы могут быть следствием стратегии декодирования, а не недостатком самой языковой модели**. В таком случае проблему можно решать на уровне декодирования, не переобучая модель.


## Блок 5. top-k и top-p

Температура меняет форму **всего** распределения, но не удаляет его длинный хвост из маловероятных токенов. Именно из этого хвоста при случайном сэмплировании могут приходить странные и малоправдоподобные продолжения.

Поэтому следующий шаг - не просто изменить распределение, а ограничить множество токенов, из которых вообще разрешено выбирать.

- **top-k** - оставляем $k$ самых вероятных токенов, остальные зануляем;
- **top-p (nucleus sampling)** - оставляем минимальный набор наиболее вероятных токенов, суммарная вероятность которых достигает порога $p$.

Главное различие:

> **top-k фиксирует размер множества кандидатов, а top-p подстраивает его размер под форму распределения.**

Если распределение очень "острое", top-p может оставить всего несколько токенов. Если оно более плоское, кандидатов станет больше.

Посмотрим на одном и том же распределении, когда эта разница действительно имеет значение.


In [ ]:
def full_dist(model, history):
    """Полное СГЛАЖЕННОЕ распределение по всему словарю (у top-k/top-p так честнее: хвост из редких токенов существует и его нужно уметь обрезать)."""
    probs = np.array([model.prob(w, history) for w in model.vocab], dtype=float)
    probs = probs / probs.sum()
    return np.array(model.vocab), probs

def show_cutoff(history, k=5, p=0.9):
    """Показываем, какие токены оставляют top-k и top-p для данного контекста."""
    words, probs = full_dist(bigram_smooth, history)
    order = np.argsort(-probs)
    words, probs = words[order], probs[order]

    k_set = set(words[:k])
    cum = np.cumsum(probs)
    cut = int(np.searchsorted(cum, p)) + 1
    p_set = set(words[:cut])

    print(f"Контекст: {history} (в словаре {len(words)} токенов)")
    print(f"{'токен':<12}{'P':>8}   top-k(k={k})   top-p(p={p})")

    for w, pr in list(zip(words, probs))[:8]:
        print(f"{w:<14}{pr:>8.3f}   "
              f"{'+' if w in k_set else '-':^9}   {'+' if w in p_set else '-':^9}")

    print(f"ИТОГО: top-k оставил {min(k,len(words))},  "
          f"top-p оставил {cut} токенов\n")

# "острый" контекст: после "на" резко доминирует пара продолжений
show_cutoff(["на"], k=5, p=0.9)

# "плоский" контекст: после "потушите" множество почти равновероятных ингредиентов
show_cutoff(["потушите"], k=5, p=0.9)

**Вопрос**

Сравните два контекста:

- на **остром распределении** после `на` доминируют всего несколько вариантов. Сколько токенов оставляют `top-p` и `top-k`?
- на **плоском распределении** после `потушите` много ингредиентов имеют похожие вероятности. Что происходит здесь?

**Возможный ответ**

На **остром** распределении почти вся вероятностная масса сосредоточена в двух токенах. `top-p` остановится после них, потому что они уже набирают нужную суммарную вероятность. А `top-k=5` в любом случае оставит пять токенов и будет вынужден добавить еще три менее вероятных варианта.

На **плоском** распределении ситуация обратная: чтобы набрать заданную массу $p$, `top-p` оставит **много токенов**, поскольку ни один из них не имеет большого веса. А `top-k=5` жестко ограничит выбор пятью вариантами, хотя десятки других могут быть почти столь же вероятными.

В этом и состоит главное преимущество `top-p`: **размер множества кандидатов адаптируется к форме распределения**. `top-k` всегда оставляет одно и то же количество токенов, поэтому на одном распределении может сохранить слишком много хвоста, а на другом - отбросить вполне разумные варианты.



In [ ]:
print("Сравнение стратегий на одной модели (по 2 запуска):\n")
configs = [
    ("чистое сэмплирование T=1.0", dict(strategy="sample", T=1.0)),
    ("top-k = 3", dict(strategy="sample", T=1.0, top_k=3)),
    ("top-p = 0.9", dict(strategy="sample", T=1.0, top_p=0.9)),
    ("T=0.8 + top-p=0.9", dict(strategy="sample", T=0.8, top_p=0.9)),
]
for name, kw in configs:
    print(name)
    for i in range(2):
        print(generate(bigram_smooth, seed=7 + i, **kw))
    print()

Температура, `top-k` и `top-p` — **не три альтернативные стратегии**, из которых нужно выбрать одну. Это независимые параметры, которые можно комбинировать.

Типичный порядок такой:

1. **Температура** меняет форму распределения — делает его более острым или более плоским.
2. **top-k / top-p** ограничивают множество кандидатов, отбрасывая хвост маловероятных токенов.
3. **Сэмплирование** выбирает следующий токен из оставшихся кандидатов.

То есть можно одновременно использовать, например:

```text
T = 0.8 + top-p = 0.9
```

Температура отвечает за степень случайности, а `top-p` — за то, насколько далеко в хвост распределения мы вообще разрешаем заглядывать.

При этом `top-k` и `top-p` тоже можно комбинировать, хотя на практике часто достаточно одного из них.



## Блок 6. Beam search

`Greedy` выбирает лучший токен **на каждом отдельном шаге**. Но локально самый вероятный выбор не обязательно приводит к самой вероятной последовательности целиком.

**Beam search** смотрит на несколько вариантов одновременно: вместо одной гипотезы он хранит $k$ лучших, на каждом шаге расширяет их и снова оставляет $k$ наиболее перспективных.

Для сравнения последовательностей используем **сумму логарифмов вероятностей**:

$$
\log P(y_1,\dots,y_n)\sum_{t=1}^{n}\log P(y_t\mid y_{<t})
$$

и нормируем ее по длине. Иначе длинные последовательности получают систематическое преимущество из-за накопления отрицательных логарифмов.




In [ ]:
def beam_search(model, k=3, max_len=20, start=None, length_norm=True):
    """Лучевой поиск. Возвращает список гипотез, отсортированных по скору."""
    init_hist = [BOS] * (model.n - 1) + (tokenize(start) if start else [])
    beams = [(init_hist, 0.0)] # (последовательность, суммарный logP)
    finished = []

    for _ in range(max_len):
        candidates = []
        for seq, score in beams:
            if seq and seq[-1] == EOS:
                finished.append((seq, score))
                continue
            dist = model.next_word_dist(seq)
            if not dist:
                dist = {w: model.prob(w, seq) for w in model.vocab}
            for w, p in sorted(dist.items(), key=lambda x: -x[1])[:k]:
                candidates.append((seq + [w], score + np.log(max(p, 1e-12))))
        if not candidates:
            break
        key = (lambda x: x[1] / max(len(x[0]), 1)) if length_norm else (lambda x: x[1])
        beams = sorted(candidates, key=key, reverse=True)[:k]

    all_beams = finished + beams
    key = (lambda x: x[1] / max(len(x[0]), 1)) if length_norm else (lambda x: x[1])
    all_beams = sorted(all_beams, key=key, reverse=True)

    def clean(seq):
        return " ".join(w for w in seq if w not in (BOS, EOS))
    return [(clean(seq), score) for seq, score in all_beams[:k]]


print("Топ-гипотезы beam search (k=4):\n")
for txt, sc in beam_search(bigram_smooth, k=4, start="обжарьте"):
    print(f"score={sc:7.3f}  {txt}")

**greedy vs beam на одной модели**


In [ ]:
g = generate(bigram_smooth, strategy="greedy", start="обжарьте")
b = beam_search(bigram_smooth, k=5, start="обжарьте")[0][0]
print("greedy:", g)
print("beam (k=5):", b)

**Вопрос**

`Beam search` стремится найти наиболее вероятную последовательность. Это хорошо работает там, где ожидается относительно однозначный или ограниченный набор правильных ответов - например, в переводе, суммаризации или распознавании речи.

А что будет с нашим генератором рецептов? Это задача с "правильным ответом" или открытая генерация?

Что мы теряем, если всегда ищем самый вероятный рецепт?

**Возможный ход рассуждений**

Генерация рецептов ближе к открытой генерации: одного правильного рецепта не существует, поэтому важны не только правдоподобность, но и разнообразие.

`Beam search` будет стремиться к одним и тем же наиболее вероятным продолжениям. В результате разные гипотезы могут оказаться очень похожими друг на друга, а менее очевидные, но интересные варианты будут отброшены.

Для открытой генерации поэтому чаще используют сэмплирование, например, с `temperature` и `top-p`, а `beam search` оставляют для задач, где поиск наиболее вероятного ответа действительно соответствует цели.



## Блок 7. Как оценивать модель: перплексия

Главная внутренняя метрика языковых моделей - перплексия:

$$
\mathrm{PP}(y_{1:M}) = \exp!\Big(-\frac{1}{M}\sum_t \log P(y_t \mid y_{<t})\Big)
$$

Это экспонента средней кросс-энтропии. Интуитивно перплексия показывает, насколько модель "удивлена" текстом: чем хуже модель объясняет последовательность, тем выше ее перплексия.

- низкая PP - модель хорошо предсказывает текст,
- высокая PP - текст для модели "неожиданный".

В простом случае равномерного распределения по словарю из $|V|$ токенов модель получает:

$$
\mathrm{PP} \approx |V|
$$

То есть она как будто каждый раз выбирает из $|V|$ вариантов с одинаковой вероятностью.

**Важное ограничение**: перплексию нельзя напрямую сравнивать между моделями с разной токенизацией или разными словарями. Если одна модель разбивает слово на несколько токенов, а другая представляет его одним токеном, значение PP будет отражать не только качество модели, но и способ токенизации.

Для нашего эксперимента это особенно удобно: все модели используют один и тот же корпус, словарь и токенизацию, поэтому их перплексии можно сравнивать напрямую.



In [ ]:
def perplexity(model, test_sentences):
    """Перплексия модели на тестовом корпусе."""
    total_logprob, n_tokens = 0.0, 0
    for s in test_sentences:
        padded = [BOS] * (model.n - 1) + s + [EOS]
        for i in range(model.n - 1, len(padded)):
            p = model.prob(padded[i], padded[:i])
            total_logprob += np.log(max(p, 1e-12))
            n_tokens += 1
    return float(np.exp(-total_logprob / n_tokens))

# честно разделим корпус на train/test
random.seed(0)
shuffled = sentences[:]
random.shuffle(shuffled)
split = int(0.8 * len(shuffled))
train_sents, test_sents = shuffled[:split], shuffled[split:]
print(f"train: {len(train_sents)} предложений, test: {len(test_sents)}")

### Сравниваем n = 1, 2, 3

Интуитивно кажется, что чем больше контекст, тем лучше модель: она видит больше информации и должна точнее предсказывать следующий токен.

Проверим это экспериментально - и посмотрим, действительно ли увеличение $n$ всегда приводит к снижению перплексии.


In [ ]:
print(f"{'модель':<12}{'|V|':>6}{'PP (train)':>14}{'PP (test)':>14}")
for n in [1, 2, 3]:
    m = LaplaceLM(train_sents, n=n, alpha=0.1)
    pp_train = perplexity(m, train_sents)
    pp_test = perplexity(m, test_sents)
    print(f"{n}-грамма   {len(m.vocab):>6}{pp_train:>14.2f}{pp_test:>14.2f}")

**Вопрос**

Почему результат оказался именно таким? На train перплексия монотонно падает с ростом $n$, а на test триграмма может оказаться даже хуже биграммы.

**Возможный ход рассуждения**

На train увеличение $n$ позволяет модели запоминать все более специфичные последовательности. Поэтому она лучше объясняет знакомые примеры и перплексия падает. Но это еще не означает, что модель лучше обобщает - здесь проявляется переобучение.

На test возникает другая проблема - разреженность. Многие триграммы, встретившиеся в тесте, могли вообще не встречаться в train. После сглаживания такие последовательности получают очень маленькую вероятность, поэтому модель сильнее "удивляется" тестовым данным и PP может вырасти.

Это напрямую соответствует тезису лекции 1: увеличение $n$ одновременно дает больше контекста и усиливает разреженность.

**Больше $n$ не означает автоматически лучшее качество.** Для конкретного корпуса существует некоторый компромисс между количеством контекста и способностью модели обобщать.



**Эксперимент со сглаживанием.** Теперь, когда у нас есть численная метрика качества, можем сделать следующий шаг - подобрать $\alpha$ не на глаз, а по перплексии на отложенных данных.

Возьмем биграммную модель и проверим несколько значений $\alpha$. Посмотрим, при каком сглаживании модель лучше всего обобщает на test.

Это уже полноценный ML-подход: гиперпараметр меняем, качество измеряем на отложенных данных и выбираем значение, которое дает лучший результат.


In [ ]:
print("Подбор alpha по перплексии на test (биграмма):\n")
best = (None, float("inf"))
for a in [0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0]:
    m = LaplaceLM(train_sents, n=2, alpha=a)
    pp = perplexity(m, test_sents)
    marker = ""
    if pp < best[1]:
        best = (a, pp)
        marker = "  <-- пока лучший"
    print(f"alpha = {a:<7} PP(test) = {pp:7.2f}{marker}")
print(f"\nЛучшая alpha = {best[0]} (PP = {best[1]:.2f})")

**Ключевое ограничение перплексии**

Низкая перплексия не означает автоматически хорошую генерацию. Перплексия оценивает, насколько хорошо модель предсказывает текст, но не то, насколько полезным, разнообразным или фактически верным окажется сгенерированный результат.

Поэтому для генеративных задач одной перплексии недостаточно. В зависимости от задачи используют:

- **BLEU / ROUGE** - когда существует эталонный ответ, например в переводе или суммаризации,
- **человеческую оценку** - по связности, релевантности, беглости и, особенно, фактической корректности.

**Вопрос**

Какие проблемы мы уже увидели сегодня, но перплексия сама по себе их не заметит? Назовите хотя бы две.

**Возможный ответ**

1. **Повторы и degeneration.** Зацикленный текст может иметь низкую перплексию: если модель постоянно повторяет один и тот же шаблон, следующий токен для нее очень предсказуем.

2. **Потеря разнообразия.** `Greedy` или `beam search` могут стабильно выдавать очень похожие тексты. Перплексия отдельного текста ничего не говорит о разнообразии множества ответов.

3. **Галлюцинации и фактические ошибки.** Перплексия показывает, насколько текст правдоподобен с точки зрения модели, но не проверяет, соответствует ли он реальности.



## Блок 8. (Опционально) Стратегии декодирования на нейросети

Все предыдущие эксперименты мы проводили на прозрачной n-граммной модели, где каждую вероятность можно было посмотреть и посчитать вручную. Но постановка задачи остается той же и для нейросетевой LM: модель по контексту вычисляет

$$
P(y_t \mid y_{<t})
$$

Меняется только способ получения этого распределения. Поэтому стратегии декодирования переносятся и на предобученную GPT.

Здесь мы проверим это на русскоязычной модели `rugpt3small`. В `transformers` те же параметры буквально представлены параметрами:

- `do_sample` - использовать сэмплирование;
- `temperature` - управлять "остротой" распределения;
- `top_k` - ограничить число кандидатов;
- `top_p` - ограничить кандидатов по суммарной вероятности;
- `num_beams` - задать ширину beam search.



In [ ]:
# !pip install -q transformers torch
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "sberbank-ai/rugpt3small_based_on_gpt2"
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
gpt = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
gpt.eval()
print("Модель загружена:", MODEL_NAME)

In [ ]:
def gpt_generate(prompt, **gen_kwargs):
    ids = tok(prompt, return_tensors="pt").input_ids
    with torch.no_grad():
        out = gpt.generate(ids, max_new_tokens=40, pad_token_id=tok.eos_token_id, **gen_kwargs)
    return tok.decode(out[0], skip_special_tokens=True)

torch.manual_seed(0)
prompt = "Чтобы приготовить борщ, сначала"
print("=== greedy (do_sample=False) ===")
print(gpt_generate(prompt, do_sample=False), "\n")

print("=== beam search (num_beams=5) ===")
print(gpt_generate(prompt, num_beams=5, do_sample=False), "\n")

print("=== T=0.7 + top_p=0.9 (рабочая связка из блока 5) ===")
print(gpt_generate(prompt, do_sample=True, temperature=0.7, top_p=0.9), "\n")

print("=== T=1.5 (высокая температура) ===")
print(gpt_generate(prompt, do_sample=True, temperature=1.5))

**Вопрос**

Сравните поведение GPT с тем, что мы наблюдали на n-граммной модели:

- дает ли `greedy` / `beam search` более предсказуемый, но однообразный текст?
- добавляет ли `top-p` разнообразия, сохраняя связность?
- становится ли текст менее связным при слишком высокой температуре?

Если наблюдения похожи, значит, главный вывод действительно перенесся с игрушечной модели на нейросетевую:

**Декодирование - это отдельный от языковой модели слой.**

Неважно, получает ли распределение простая биграмма или нейросеть с миллиардами параметров: после получения $P(y_t \mid y_{<t})$ логика выбора следующего токена остается той же.



## Итоги

- **Разделяйте модель и декодирование.** Мы меняли поведение генерации, не переобучая модель: за предсказуемость и разнообразие отвечает стратегия декодирования. Если генерация "не та", сначала проверьте параметры декодирования, а уже потом меняйте модель.

- **Разреженность - реальная проблема частотных моделей.** Мы получили $P=0$ даже для грамматичной фразы и увидели, зачем нужно сглаживание. Любая частотная оценка должна уметь работать с сочетаниями, которые модель не видела.

- **Больше контекста не всегда значит лучше.** Триграмма могла проиграть биграмме на test из-за разреженности. Точно так же увеличение ширины beam не гарантирует более качественный текст. Любое усложнение нужно проверять на отложенных данных.

- **Стратегию декодирования выбирают под задачу.** Для задач с относительно определенным правильным ответом подходят beam search и более низкая температура. Для открытой генерации, где важно разнообразие, чаще используют сэмплирование с умеренной температурой и `top-p`. Универсально лучшей стратегии нет.

- **"Более вероятный" не значит "более хороший".** Beam может найти наиболее вероятную последовательность, но она окажется шаблонной. Перплексия хорошо измеряет предсказуемость текста, но не замечает повторы, потерю разнообразия и галлюцинации.

- **Автоматические метрики не заменяют оценку результата.** Для генеративных задач важно смотреть не только на вероятности и перплексию, но и на связность, релевантность, разнообразие и фактическую корректность текста.




## Вопросы для рефлексии

1. Почему униграмма не отличила осмысленную фразу от бессмысленной, а биграмма смогла? Что именно дает учет контекста?

2. Приведите пример грамматичной фразы, которой ваша биграммная модель припишет нулевую вероятность без сглаживания. Как сглаживание решает эту проблему и какой ценой?

3. При каком значении $T$ сэмплирование приближается к `greedy`, а при каком - к равномерному случайному выбору? Почему?

4. Чем `top-p` принципиально отличается от `top-k`? Приведите пример контекста, в котором фиксированный `top-k=5` работает плохо.

5. Почему триграмма может проиграть биграмме по перплексии на test, хотя на train показывает лучший результат?

6. Назовите проблему генерации, которую перплексия не замечает, и объясните, почему она для нее невидима.
